In [1]:
# ================================================================
# CELL 1: SETUP & IMPORTS (VERSION 12 META-ENSEMBLE)
# ================================================================
!pip install -q catboost lightgbm scipy scikit-learn pandas numpy

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import f1_score, roc_auc_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from scipy.optimize import minimize
import lightgbm as lgb
import catboost as cb

print("Version 12 Meta-Ensemble Environment Ready.")

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Version 12 Meta-Ensemble Environment Ready.


In [4]:
# ================================================================
# CELL 2: LOAD V11 OOF PREDICTIONS AND GROUND TRUTH (FIXED)
# ================================================================
CHKPT_DIR = Path('./chkpt_v11') # Update if your path is different
CACHE_DIR = Path('./cache_v11') # Update if your path is different

# 1. Load Ground Truth Safely
found_csv = False
for search_dir in [Path('..'), Path('.')]:
    for p in search_dir.rglob('train.csv'):
        # Skip hidden system directories that Kaggle creates
        if '.virtual_documents' in str(p) or '.cache' in str(p): 
            continue
            
        try:
            df = pd.read_csv(p)
            print(f"👀 Found file: {p}")
            print(f"   -> Columns available: {df.columns.tolist()}")
            
            # Smart target column detection
            possible_targets = ['label', 'target', 'is_fake', 'fake', 'class', 'ground_truth']
            target_col = None
            for col in possible_targets:
                if col in df.columns:
                    target_col = col
                    break
            
            if target_col is not None:
                y_all = df[target_col].values
                print(f"✅ Loaded {len(y_all)} ground truth labels from the '{target_col}' column!\n")
                found_csv = True
                break
            else:
                print(f"⚠️ Skipping this file (No target column found).\n")
        except Exception as e:
            pass
            
    if found_csv: break

if not found_csv: 
    raise ValueError("🚨 ERROR: Cannot find the correct train.csv with a valid target column.")

# 2. Load Level 1 OOF & Test Predictions
base_models = ['dino_ft', 'siglip_ft', 'clip_ft', 'xgb']
oof_store = {}
test_store = {}

for m in base_models:
    # Notice I used the exact naming from your V11 saves (e.g. 'sig_oof.npy')
    file_prefix = 'sig' if m == 'siglip_ft' else 'clip' if m == 'clip_ft' else 'dino' if m == 'dino_ft' else 'xgb'
    
    oof_path = CHKPT_DIR / f"{file_prefix}_oof.npy"
    test_path = CHKPT_DIR / f"{file_prefix}_test.npy"
    
    if oof_path.exists():
        oof_store[m] = np.load(oof_path)
        print(f"✅ Loaded {m} OOF: AUC = {roc_auc_score(y_all, oof_store[m]):.4f}")
    if test_path.exists():
        test_store[m] = np.load(test_path)

print("CELL 2 COMPLETE")

👀 Found file: ../data/DCU 2026 ML challenge - external 2/train.csv
   -> Columns available: ['image_id', 'ground_truth']
✅ Loaded 4800 ground truth labels from the 'ground_truth' column!

✅ Loaded dino_ft OOF: AUC = 0.9627
✅ Loaded siglip_ft OOF: AUC = 0.9582
✅ Loaded clip_ft OOF: AUC = 0.9600
CELL 2 COMPLETE


In [5]:
# ================================================================
# CELL 3: MEGA-CONCATENATION & LEVEL-2 BOOSTERS
# Automatically finds cached .npy features and trains CatBoost & LightGBM
# ================================================================
print("Searching for cached Level-1 features...")
train_features = []
test_features = []

# Dynamically load all available feature arrays
for p in CACHE_DIR.glob('*_train.npy'):
    model_name = p.stem.replace('_train', '')
    test_p = CACHE_DIR / f"{model_name}_test.npy"
    
    if test_p.exists():
        print(f"  -> Found feature pair for: {model_name}")
        train_features.append(np.load(p))
        test_features.append(np.load(test_p))

if len(train_features) > 0:
    X_meta = np.hstack(train_features)
    X_meta_test = np.hstack(test_features)
    print(f"✅ Created Mega-Concatenation vector. Shape: {X_meta.shape}")
    
    # Setup CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cat_oof = np.zeros(len(y_all)); cat_test = np.zeros((5, len(X_meta_test)))
    lgb_oof = np.zeros(len(y_all)); lgb_test = np.zeros((5, len(X_meta_test)))
    
    print("\nTraining CatBoost & LightGBM on Mega-Vector...")
    for fold, (ti, vi) in enumerate(skf.split(X_meta, y_all)):
        # Train LightGBM
        clf_lgb = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42, n_jobs=-1)
        clf_lgb.fit(X_meta[ti], y_all[ti], eval_set=[(X_meta[vi], y_all[vi])], callbacks=[lgb.early_stopping(50, verbose=False)])
        lgb_oof[vi] = clf_lgb.predict_proba(X_meta[vi])[:, 1]
        lgb_test[fold] = clf_lgb.predict_proba(X_meta_test)[:, 1]
        
        # Train CatBoost
        clf_cat = cb.CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_state=42)
        clf_cat.fit(X_meta[ti], y_all[ti], eval_set=(X_meta[vi], y_all[vi]), early_stopping_rounds=50)
        cat_oof[vi] = clf_cat.predict_proba(X_meta[vi])[:, 1]
        cat_test[fold] = clf_cat.predict_proba(X_meta_test)[:, 1]
        
    oof_store['lgbm'] = lgb_oof; test_store['lgbm'] = lgb_test.mean(axis=0)
    oof_store['cat'] = cat_oof; test_store['cat'] = cat_test.mean(axis=0)
    
    print(f"✅ LightGBM OOF AUC: {roc_auc_score(y_all, lgb_oof):.4f}")
    print(f"✅ CatBoost OOF AUC: {roc_auc_score(y_all, cat_oof):.4f}")
else:
    print("⚠️ No cached .npy features found. Skipping Level-2 Boosters.")

print("CELL 3 COMPLETE")

Searching for cached Level-1 features...
  -> Found feature pair for: clip
  -> Found feature pair for: forensic
  -> Found feature pair for: cnn
  -> Found feature pair for: siglip
  -> Found feature pair for: dino
✅ Created Mega-Concatenation vector. Shape: (4800, 4082)

Training CatBoost & LightGBM on Mega-Vector...
[LightGBM] [Info] Number of positive: 1852, number of negative: 1988
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.411392 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1032688
[LightGBM] [Info] Number of data points in the train set: 3840, number of used features: 4055
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.482292 -> initscore=-0.070863
[LightGBM] [Info] Start training from score -0.070863
[LightGBM] [Info] Number of positive: 1852, number of negative: 1988
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.420613 seconds.
You can se

In [6]:
# ================================================================
# CELL 4: NELDER-MEAD CONTINUOUS OPTIMIZER
# Finds the mathematically perfect floating-point weights
# ================================================================
print("Running Nelder-Mead Optimization...")

# Prepare matrix of all OOF predictions
active_models = list(oof_store.keys())
P_oof = np.vstack([oof_store[m] for m in active_models]).T

# We minimize Log Loss to find perfectly calibrated weights
def objective(weights):
    weights = np.clip(weights, 0, 1) # Prevent negative weights
    if np.sum(weights) == 0: return 999
    weights /= np.sum(weights) # Normalize to 1
    blend = np.dot(P_oof, weights)
    return log_loss(y_all, blend)

# Initial guess (equal weights)
init_weights = [1.0 / len(active_models)] * len(active_models)

# Run Optimizer
res = minimize(objective, init_weights, method='Nelder-Mead', options={'maxiter': 1000})

# Get best weights
optimal_weights = np.clip(res.x, 0, 1)
optimal_weights /= np.sum(optimal_weights)

# Apply weights
nelder_oof = np.dot(P_oof, optimal_weights)

# Find Best Threshold for F1
nm_best_f1, nm_best_t = 0, 0.5
for t in np.arange(0.3, 0.7, 0.01):
    f = f1_score(y_all, (nelder_oof >= t).astype(int))
    if f > nm_best_f1: nm_best_f1 = f; nm_best_t = t

print(f"\n🏆 NELDER-MEAD OPTIMAL WEIGHTS:")
for m, w in zip(active_models, optimal_weights):
    print(f"  {m:<10}: {w:.4f} ({w*100:.1f}%)")
print(f"🔥 Nelder-Mead Ensemble OOF F1: {nm_best_f1:.4f} @ Thr={nm_best_t:.2f}")

print("CELL 4 COMPLETE")

Running Nelder-Mead Optimization...

🏆 NELDER-MEAD OPTIMAL WEIGHTS:
  dino_ft   : 0.7117 (71.2%)
  siglip_ft : 0.2883 (28.8%)
  clip_ft   : 0.0000 (0.0%)
  lgbm      : 0.0000 (0.0%)
  cat       : 0.0000 (0.0%)
🔥 Nelder-Mead Ensemble OOF F1: 0.9163 @ Thr=0.46
CELL 4 COMPLETE


In [7]:
# ================================================================
# CELL 5: META-LEARNER (L2 LOGISTIC REGRESSION)
# ================================================================
print("Training L2 Logistic Regression Meta-Learner...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
meta_oof = np.zeros(len(y_all))

# Predict Test Set if test_store has all active models
can_predict_test = all(m in test_store for m in active_models)
if can_predict_test:
    P_test = np.vstack([test_store[m] for m in active_models]).T
    meta_test_preds = np.zeros((5, len(P_test)))

for fold, (ti, vi) in enumerate(skf.split(P_oof, y_all)):
    # Ridge classifier allows the Meta-Learner to learn conditional logic
    meta_model = LogisticRegression(C=1.0, penalty='l2', max_iter=1000, class_weight='balanced')
    meta_model.fit(P_oof[ti], y_all[ti])
    meta_oof[vi] = meta_model.predict_proba(P_oof[vi])[:, 1]
    
    if can_predict_test:
        meta_test_preds[fold] = meta_model.predict_proba(P_test)[:, 1]

# Find Best Threshold
meta_best_f1, meta_best_t = 0, 0.5
for t in np.arange(0.3, 0.7, 0.01):
    f = f1_score(y_all, (meta_oof >= t).astype(int))
    if f > meta_best_f1: meta_best_f1 = f; meta_best_t = t

print(f"🤖 Meta-Learner OOF F1: {meta_best_f1:.4f} @ Thr={meta_best_t:.2f}")

# Decide the winner
print("\n--- STRATEGY COMPARISON ---")
print(f"Nelder-Mead : {nm_best_f1:.4f}")
print(f"Meta-Learner: {meta_best_f1:.4f}")

if nm_best_f1 > meta_best_f1:
    print("\n👑 WINNER: NELDER-MEAD OPTIMIZATION")
    final_test_preds = np.dot(P_test, optimal_weights)
    final_threshold = nm_best_t
else:
    print("\n👑 WINNER: META-LEARNER")
    final_test_preds = meta_test_preds.mean(axis=0)
    final_threshold = meta_best_t

print("CELL 5 COMPLETE")

Training L2 Logistic Regression Meta-Learner...
🤖 Meta-Learner OOF F1: 0.9287 @ Thr=0.41

--- STRATEGY COMPARISON ---
Nelder-Mead : 0.9163
Meta-Learner: 0.9287

👑 WINNER: META-LEARNER
CELL 5 COMPLETE


In [11]:
# ================================================================
# FINAL SUBMISSION GENERATOR (V12)
# ================================================================
import pandas as pd
import numpy as np
from pathlib import Path

print("Generating Final V12 Submission using Nelder-Mead Weights...")

# 1. Load the test predictions directly from disk
dino_test = np.load('./chkpt_v11/dino_test.npy')
sig_test = np.load('./chkpt_v11/sig_test.npy')

# 2. Apply your winning Nelder-Mead calculus weights!
final_test_probs = (dino_test * 0.7117) + (sig_test * 0.2883)

# 3. Apply your optimal threshold
final_test_preds = (final_test_probs >= 0.46).astype(int)

# 4. Find the template (test.csv or your old submission.csv)
template_path = None
search_paths = [
    Path('../data/DCU 2026 ML challenge - external 2/test.csv'),
    Path('./data/submission.csv'),
    Path('../data/submission.csv'),
    Path('../data/DCU 2026 ML challenge - external 2/submission.csv')
]

for p in search_paths:
    if p.exists():
        template_path = p
        print(f"👀 Found template file at: {p}")
        break

if template_path:
    # Read the template
    template_df = pd.read_csv(template_path)
    
    # Check what the ID column is named (usually 'image_id' or 'id')
    id_col = 'image_id' if 'image_id' in template_df.columns else template_df.columns[0]
    
    # Build the final clean dataframe
    # Based on your train set, the competition evaluates the 'ground_truth' column
    final_sub = pd.DataFrame({
        id_col: template_df[id_col],
        'ground_truth': final_test_preds
    })
    
    # Save it!
    final_sub.to_csv('submission_v12.csv', index=False)
    
    print("\n✅ SUCCESS! Saved submission_v12.csv")
    print(f"Total predictions: {len(final_test_preds)}")
    print("\nClass Balance Predicted (0=Real, 1=Fake):")
    print(final_sub['ground_truth'].value_counts(normalize=True) * 100)
    
else:
    print("\n🚨 ERROR: Could not find test.csv or submission.csv. Please verify the path to your test file!")

Generating Final V12 Submission using Nelder-Mead Weights...
👀 Found template file at: ../data/DCU 2026 ML challenge - external 2/test.csv

✅ SUCCESS! Saved submission_v12.csv
Total predictions: 2058

Class Balance Predicted (0=Real, 1=Fake):
ground_truth
1    51.506317
0    48.493683
Name: proportion, dtype: float64


In [10]:
import os
from pathlib import Path

print("🔍 CHECKING CHECKPOINT FOLDER (chkpt_v11):")
chkpt_path = Path('./chkpt_v11')
if chkpt_path.exists():
    for f in sorted(os.listdir(chkpt_path)):
        print(f"  - {f}")
else:
    print("  ⚠️ Folder not found!")

print("\n🔍 CHECKING CACHE FOLDER (cache_v11):")
cache_path = Path('./cache_v11')
if cache_path.exists():
    for f in sorted(os.listdir(cache_path)):
        print(f"  - {f}")
else:
    print("  ⚠️ Folder not found!")

🔍 CHECKING CHECKPOINT FOLDER (chkpt_v11):
  - clip_meta.json
  - clip_oof.npy
  - clip_test.npy
  - dino_meta.json
  - dino_oof.npy
  - dino_test.npy
  - sig_meta.json
  - sig_oof.npy
  - sig_test.npy

🔍 CHECKING CACHE FOLDER (cache_v11):
  - clip_test.npy
  - clip_train.npy
  - cnn_test.npy
  - cnn_train.npy
  - dino_test.npy
  - dino_train.npy
  - forensic_test.npy
  - forensic_train.npy
  - siglip_test.npy
  - siglip_train.npy
